# Problem Set: Multilingual review sheets for a zoo alliance
### Chapter 9 — Ontologies and Natural Languages

**Topic coverage:** Keet §9.1 (multilingual ontologies, lexicons with morphology), §9.2 (verbalisation into controlled natural language, and parsing it back); course notebooks 01–03 of this chapter. Tools: DSPy + GEPA, an LLM judge validated against human labels, LangChain agents, and best-of-n resampling as an optimal-stopping MDP.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 18 / Part B 36 / Part C 24 / Part D 22); 54 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$8–16 *estimated* on `claude-opus-5` for a full solutions run (≈ 330 DSPy calls at ≈ \$0.02–0.04 each, plus ≈ \$1–2 for ten agent episodes; the two GEPA runs and the noise study are the largest lines). Nothing here has been measured — your cost lines are the measurement. Re-running unchanged cells is free: DSPy caches identical requests, except where a problem asks for `cache=False`. Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

The **Rhine–Meuse Zoo Alliance** — five zoos in the Netherlands and Germany plus an
English-speaking partner — keeps one shared animal-husbandry ontology: species,
diets, enclosures, named animals. Every quarter the **Husbandry Standards
Committee** has the axioms reviewed by the keepers who work with the animals, each
in their own language. Keepers do not read OWL, so every axiom is *verbalised*
into a controlled sentence on a review sheet ("Jedes Okapi lebt in mindestens
einem Gehege."). A sentence that says something other than the axiom gets a wrong
rule signed off; a sentence a keeper cannot read gets rewritten by hand, which
costs the committee about \$1.50 of keeper time per sentence.

You are the ontology engineer asked to replace the translator with a Claude
verbaliser. Your brief:

1. **Grade it without a model where you can.** Verbalisation has an inverse: parse
   the sentence back and compare axioms. That round trip is free and exact — and
   it cannot see whether a keeper will accept the sentence. Build the grader from
   both halves, and audit the lexicon the round trip depends on.
2. **Grade readability with a judge you have validated.** The keepers labelled 22
   sentences. An LLM judge is only allowed onto the review pipeline if it agrees
   with them better than the committee's regex style checks do.
3. **Build and optimise the verbaliser honestly** — train / dev / test split by
   axiom, a GEPA budget, the cost of every run, run-to-run noise, and one run
   that shows what happens when you optimise the exact half alone.
4. **Decide when to stop resampling.** Model best-of-n drafting as an
   optimal-stopping MDP, derive the stopping rule, calibrate it with live samples,
   and ship an agent that checks its own drafts before it answers.

The lexicon (en/nl/de, with morphology), the controlled grammar and its parser,
the committee's style checks, the corpus (15 axioms × 2 languages = 30 items,
split 5/5/5 axioms), the keepers' readability panel and the resampling MDP are
provided in `ch09_agentic.py`.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `course/artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `04_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root and this chapter importable no matter where Jupyter started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import oe_course
from oe_course import config
from oe_course.grading import Grader

print(json.dumps(oe_course.describe_environment(), indent=1))
config.require_credentials()          # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 9 — Ontologies and Natural Languages — Multilingual review sheets for a zoo alliance')

import re, random, statistics as st
import dspy
import numpy as np
import pandas as pd
import ch09_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt, agents

print(A.validate())                                   # every gold artefact re-checked
train, dev, test = A.build_dataset("train"), A.build_dataset("dev"), A.build_dataset("test")
ALL = {ex.id: ex for ex in train + dev + test}
print(f"train {len(train)} / dev {len(dev)} / test {len(test)} items "
      f"(split by axiom: {len({ex.axiom_id for ex in test})} test axioms)")
print(test[1].axiom, f"[{test[1].language}]")
print(test[1].terms)

---

## Part A — The free grader, and what it cannot see

`A.round_trip(sentence, axiom, language)` parses a sentence with the controlled
grammar and compares the recovered axiom with the intended one. `A.lint(...)` runs the
committee's regex style checks. Read `A.STYLE_GUIDE` and the docstrings of `parse_cnl`
and `lint` before you start: both are *deliberately* imperfect in the ways real ones are.

In [ ]:
for language, patterns in A.STYLE_GUIDE.items():
    print(language)
    for construct, pattern in patterns.items():
        print(f"   {construct:11s} {pattern}")
ex = ALL["okapi-lives-in-enclosure-de"]
print("\n", ex.reference, "->", A.round_trip(ex.reference, A.gold_axiom(ex), "de"))

### A1. Faithful is not the same as acceptable (10 pts)

For **three** corpus items of your choice (keys of `ALL`, at least two languages), write
two sentences each:

* `"faithful-but-bad"` — round-trips **exactly** to the item's axiom, but a keeper
  would reject it (wrong article, gender, number, case, capitalisation, a leaked
  identifier, a word from the wrong language, …). Not the reference sentence.
* `"fluent-but-unparseable"` — correct, natural language that a keeper would accept
  but the round trip rejects (it does not parse, or parses to another axiom).

Build a DataFrame `a1` with columns `item`, `kind`, `sentence`, `faithful`
(`A.round_trip(...)["faithful"]`) and `lint_issues` (the rule ids `A.lint` reports).
At least one of your faithful-but-bad sentences must pass the lint with **no** issue.

Then answer in writing: (a) why the round trip cannot be the whole metric, and why the
lint cannot be the rest of it; (b) the alliance could hand-write a template verbaliser
instead of using Claude — when is the LLM worth it, and what makes it safe to use here?

In [ ]:
candidates = {
    # "item-id": {"faithful-but-bad": "...", "fluent-but-unparseable": "..."},
}
# TODO: fill `candidates` for three items, then build the DataFrame `a1`.
a1 = None

raise NotImplementedError

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=6):
    assert isinstance(a1, pd.DataFrame) and len(candidates) >= 3
    assert {"item", "kind", "sentence", "faithful", "lint_issues"} <= set(a1.columns)
    assert len({ALL[i].language for i in candidates}) >= 2, "use at least two languages"
    norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
    for item, pair in candidates.items():
        ex = ALL[item]
        axiom = A.gold_axiom(ex)
        bad, fluent = pair["faithful-but-bad"], pair["fluent-but-unparseable"]
        assert A.round_trip(bad, axiom, ex.language)["faithful"], f"{item}: 'faithful-but-bad' must round-trip"
        assert norm(bad) != norm(ex.reference), f"{item}: do not reuse the reference sentence"
        assert not A.round_trip(fluent, axiom, ex.language)["faithful"], f"{item}: the fluent one round-trips"
    for _, row in a1.iterrows():
        ex = ALL[row["item"]]
        assert row["faithful"] == A.round_trip(row["sentence"], A.gold_axiom(ex), ex.language)["faithful"]
    clean = [p["faithful-but-bad"] for i, p in candidates.items()
             if not A.lint(p["faithful-but-bad"], A.gold_axiom(ALL[i]), ALL[i].language)]
    assert clean, "find a faithful-but-bad sentence that the lint does not flag"

**Your answer:**

*(Write your answer here.)*

### A2. Audit the translation agency's delivery (8 pts)

The round trip is only as good as the lexicon it parses with. The agency's first Dutch
delivery is `A.NL_DELIVERY_V0`; `A.lexicon_with("nl", A.NL_DELIVERY_V0)` is the lexicon
with it installed.

(a) Implement `audit_lexicon(lexicon, language) -> set[tuple[str, str]]` returning
`(kind, term)` pairs for four kinds of defect, using `A.forms(term, language, lexicon)`:

| kind | meaning |
|---|---|
| `"missing"` | the term has no label in this language |
| `"collision"` | a surface form of the term is also a form of another term (report **both** terms) |
| `"identifier"` | a label looks like a camelCase identifier (`[a-z][A-Z]`) |
| `"template-word"` | a label contains, as a whole word, one of `A.TEMPLATE_WORDS[language]` |

(b) Build `a2`: one row per Dutch sentence in the corpus references and in
`A.READABILITY_PANEL`, with columns `source` (item or panel id), `sentence`,
`faithful_before` (with `A.LEXICON`) and `faithful_after` (with the delivery).

In writing: which defect broke which sentences, and which kind of defect is the most
dangerous in production, and why?

In [ ]:
def audit_lexicon(lexicon: dict, language: str) -> set[tuple[str, str]]:
    # TODO (a)
    raise NotImplementedError

delivery = A.lexicon_with("nl", A.NL_DELIVERY_V0)
a2 = None       # TODO (b)

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=5):
    for language in A.LANGUAGES:
        assert audit_lexicon(A.LEXICON, language) == set(), f"false alarm on the reviewed {language} lexicon"
    assert audit_lexicon(A.lexicon_with("nl", A.NL_DELIVERY_V0), "nl") == {
        ("missing", "Insectivore"), ("collision", "Enclosure"), ("collision", "Paddock"),
        ("identifier", "caresFor"), ("template-word", "Okapi")}
    toy = {"Foo": {"kind": "class", "en": {"label": "only child"}},
           "Bar": {"kind": "class", "en": {"label": "bar", "plural": "bars"}},
           "Baz": {"kind": "class", "en": {"label": "Bars"}}}
    assert audit_lexicon(toy, "en") == {("template-word", "Foo"), ("collision", "Bar"), ("collision", "Baz")}
    assert isinstance(a2, pd.DataFrame) and len(a2) == 15
    assert {"source", "sentence", "faithful_before", "faithful_after"} <= set(a2.columns)
    assert a2["faithful_before"].all()
    _d = A.lexicon_with("nl", A.NL_DELIVERY_V0)
    _ax = {**{f"{aid}-nl": ax for aid, _, ax, refs in A.CORPUS if "nl" in refs},
           **{p["id"]: A.AXIOMS[p["axiom_id"]] for p in A.READABILITY_PANEL}}
    for _, row in a2.iterrows():
        assert row["faithful_after"] == A.round_trip(row["sentence"], _ax[row["source"]], "nl", _d)["faithful"]

**Your answer:**

*(Write your answer here.)*

---

## Part B — A two-part grader, a Claude verbaliser, and a judge you can trust

The guidelines a scorer (or judge) can report are the committee's style guide, as
`A.CNL_RULEBOOK`. Note the last three: no regex in this project can report them.

In [ ]:
for rule in A.CNL_RULEBOOK:
    print(f"{rule.id:26s} {rule.description[:95]}")

### B1. The staged scorer: exact half, style half (12 pts)

Implement `verbalisation_scorer(gold, pred) -> ev.ScoreReport`, where `gold` is a dataset
row (use `A.gold_axiom(gold)`, `gold.language`) and `pred.sentence` is the model's
sentence. Stages, in order:

| outcome | score | `violated` |
|---|---|---|
| empty, or `A.parse_cnl` returns `None` | 0.0 | `[f"template-for-{operator}"]` |
| parses, but to a different **construct** | 0.25 | `[f"template-for-{operator}"]` |
| parses to the right construct with different **terms** | 0.25 | `["preserve-the-terms"]` |
| faithful | `0.5 + 0.5 * max(0, 1 - 0.25 * k)` | the `k` distinct rule ids `A.lint` reports, first-seen order |

So a faithful sentence earns the exact half outright, and the style half loses a quarter
per distinct kind of lint issue. Unfaithful sentences earn no style credit — a readable
sentence that says the wrong thing is worse than useless on a review sheet. Put the
recovered axiom in `notes` when it differs (the words "recovered" and the axiom), and the
lint messages when there are any: that text is what GEPA's reflection reads.

In [ ]:
def verbalisation_scorer(gold, pred) -> ev.ScoreReport:
    sentence = str(getattr(pred, "sentence", "") or "").strip()
    axiom, language = A.gold_axiom(gold), gold.language
    # TODO: the four stages from the table, in order.
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=8):
    P = lambda s: dspy.Prediction(sentence=s)
    de, en = ALL["okapi-lives-in-enclosure-de"], ALL["tembo-is-elephant-en"]
    cases = [
        (de, "Jedes Okapi lebt in mindestens einem Gehege.", 1.0, []),
        (de, "", 0.0, ["template-for-some"]),
        (de, "Okapis leben in Gehegen.", 0.0, ["template-for-some"]),
        (de, "Jedes Okapi ist ein Gehege.", 0.25, ["template-for-some"]),
        (de, "Jedes Okapi lebt in mindestens einem Stall.", 0.25, ["preserve-the-terms"]),
        (de, "Jedes Okapi lebt in mindestens ein Gehege.", 1.0, []),          # the blind spot
        (de, "Jeder Okapi lebt in mindestens einem Gehege.", 0.875, ["agree-in-gender"]),
        (de, "Jeder Okapi livesIn mindestens einem Gehege.", 0.75,
         ["use-the-lexicon-label", "agree-in-gender"]),
        (en, "Tembo is an elephant.", 1.0, []),
        (en, "Tembo is a elephant.", 0.875, ["correct-article"]),
        (en, "Every tembo is an elephant.", 0.0, ["template-for-type"]),
    ]
    for gold, sentence, score, violated in cases:
        r = verbalisation_scorer(gold, P(sentence))
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (sentence, r.score, r.violated)
        assert all(v in A.CNL_RULEBOOK for v in r.violated)
    wrong = verbalisation_scorer(de, P("Jedes Okapi ist ein Gehege."))
    assert any("recovered" in n.lower() and "Okapi SubClassOf Enclosure" in n for n in wrong.notes)
    for ex in train + dev + test:
        assert verbalisation_scorer(ex, P(ex.reference)).score == 1.0, ex.id

### B2. The verbaliser as a DSPy program (6 pts)

Write a signature `Verbalisation` with inputs `axiom`, `language`, `terms` and output
`sentence`, and a factory `Verbaliser(instruction)` returning a `dspy.Module` with a
single `dspy.Predict` whose instruction is `instruction`. The field descriptions are part
of the prompt (manual marks): say what a keeper needs, not what DSPy needs. Do **not**
put the style guide into the signature — the baseline must not know the templates; that
is what Part C optimises. Configure Claude and run the program once on `train[0]`.

In [ ]:
BASELINE_INSTRUCTION = ("Write the axiom as one sentence in the requested language, "
                        "for a zoo keeper's review sheet.")

# TODO: class Verbalisation(dspy.Signature): ...
# TODO: def Verbaliser(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: Verbaliser()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=3):
    assert set(Verbalisation.input_fields) == {"axiom", "language", "terms"}
    assert "sentence" in Verbalisation.output_fields
    program = Verbaliser("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert isinstance(smoke.sentence, str) and smoke.sentence.strip()

### B3. Baseline on the development split, with its cost (6 pts)

Evaluate the baseline on `dev` with `ev.evaluate_dataset` inside `llm.meter(lm)`; store
the result in `baseline_dev` and the cost in `baseline_dev_cost`, and show the per-item
rows **with the sentences** (run the program yourself or re-run — it is cached).

Write a short error analysis: which stage does each non-perfect item fail at (parse,
meaning, style), and is it a language problem or a format problem? Which of the
committee's rules could the scorer *not* have told you about?

In [ ]:
# TODO: baseline_dev = ..., baseline_dev_cost = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=2):
    assert baseline_dev["n"] == len(dev) == 10
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert 0.0 <= baseline_dev["mean_score"] <= 1.0
    assert {"calls", "usd"} <= set(baseline_dev_cost)

**Your answer:**

*(Write your answer here.)*

### B4. Validate a readability judge against the keepers (12 pts)

`A.READABILITY_PANEL` holds 22 faithful sentences with the keepers' majority label
(`acceptable`). Before a judge scores anything that matters, it has to agree with them.

1. Write a signature `ReadabilityVerdict` (inputs `sentence`, `language`, `terms`;
   outputs `acceptable: bool` and `problem: str`) and `judge_program =
   dspy.Predict(ReadabilityVerdict)`. Give it the term sheet (`A.term_sheet(axiom,
   language)`), never the human label. Write the instruction as the committee would brief
   a new reviewer. Wrap it as `judge_sentence(sentence, language, axiom) -> bool` (C2
   reuses it).
2. Implement `cohens_kappa(a, b)` for two equal-length boolean sequences
   (κ = (p_o − p_e)/(1 − p_e); return 1.0 when p_e = 1 and the raters agree fully).
3. Run the judge on the panel inside `llm.meter(lm)` (cost in `b4_cost`). Build `b4` with
   columns `id`, `language`, `human`, `judge`, `lint_ok` (no `A.lint` issue) and
   `b4_stats = {"judge": {"accuracy": …, "kappa": …}, "lint": {…}}`, both against
   `human`. Set `judge_trusted = kappa_judge >= 0.6 and kappa_judge > kappa_lint`.

In writing: where does the judge disagree with the keepers — and is it the judge that is
wrong? Would you put this judge into the GEPA metric, into the test report only, or
nowhere, and what would you need first?

In [ ]:
# TODO 1: class ReadabilityVerdict(dspy.Signature): ...
judge_program = None

def judge_sentence(sentence: str, language: str, axiom) -> bool:
    # TODO 1
    raise NotImplementedError

def cohens_kappa(a, b) -> float:
    # TODO 2
    raise NotImplementedError

# TODO 3: b4, b4_stats, b4_cost, judge_trusted

In [ ]:
# Marking scheme for B4 -- do not edit.
with GRADER.check('B4', points=6):
    assert abs(cohens_kappa([1, 1, 0, 0], [1, 0, 0, 0]) - 0.5) < 1e-9
    assert abs(cohens_kappa([True, False, True, False], [False, True, False, True]) + 1.0) < 1e-9
    assert abs(cohens_kappa([1, 1, 1, 1], [1, 0, 1, 0])) < 1e-9
    assert cohens_kappa([1, 0, 1], [1, 0, 1]) == 1.0 and cohens_kappa([1, 1], [1, 1]) == 1.0
    assert set(ReadabilityVerdict.input_fields) == {"sentence", "language", "terms"}
    assert {"acceptable", "problem"} <= set(ReadabilityVerdict.output_fields)
    assert callable(judge_sentence)
    assert isinstance(b4, pd.DataFrame) and len(b4) == len(A.READABILITY_PANEL)
    assert {"id", "language", "human", "judge", "lint_ok"} <= set(b4.columns)
    assert list(b4.human) == [p["acceptable"] for p in A.READABILITY_PANEL], "human column = panel labels"
    assert b4.lint_ok.sum() == 17, "lint_ok must be 'A.lint reports nothing'"
    for who, col in (("judge", "judge"), ("lint", "lint_ok")):
        assert abs(b4_stats[who]["accuracy"] - (b4[col] == b4.human).mean()) < 1e-3
        assert abs(b4_stats[who]["kappa"] - cohens_kappa(b4[col], b4.human)) < 1e-3
    _k = b4_stats["judge"]["kappa"]
    assert judge_trusted == (_k >= 0.6 and _k > b4_stats["lint"]["kappa"])
    assert {"calls", "usd"} <= set(b4_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise the verbaliser — and report it honestly

Rules for this part: **optimise on `train`, select on `dev`, report on `test`**, and put a
cost and a noise estimate next to every number. The test split holds five axioms that no
language version of has been seen before.

### C1. A budgeted GEPA run with a held-out report (10 pts)

1. Build the GEPA metric from your scorer and `A.CNL_RULEBOOK`, and a reflection LM.
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Compare baseline and tuned on **`test`** with `opt.compare` (store as `c1`).
4. Save the tuned instruction to `config.artifacts_dir() / "ch09_verbaliser_instruction.txt"`.

In writing: read the instruction diff. Which templates and style rules did GEPA write
down, which did it miss — in particular the last three rules of the rulebook — and why?

In [ ]:
from oe_course import config
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = config.artifacts_dir() / "ch09_verbaliser_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=5):
    axioms_of = lambda rows: {ex.axiom_id for ex in rows}
    assert not (axioms_of(test) & (axioms_of(train) | axioms_of(dev))), "test axioms must be unseen"
    assert {r["item"] for r in c1.after["rows"]} == {ex.id for ex in test}, "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test) == 10
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 200

**Your answer:**

*(Write your answer here.)*

### C2. Optimise the exact half alone, and look at what you get (8 pts)

1. Implement `fidelity_only(gold, pred) -> ev.ScoreReport`: 1.0 if the round trip is
   faithful, else 0.0 with `violated=[f"template-for-{operator}"]` — no style half.
2. Run GEPA with it (`FIDELITY_BUDGET = 40`, same splits, inside `llm.meter` →
   `fidelity_gepa_cost`); call the result `tuned_fidelity`.
3. On `test`, build `c2` with one row per program (`"gepa"` from C1 and
   `"gepa-fidelity-only"`) and columns `program`, `fidelity`, `full_score`,
   `judge_acceptable` (share of the program's test sentences your B4 judge accepts),
   `eval_usd` and `optimisation_usd`. Generate each program's sentences once and score
   them three ways.

In writing: did optimising the exact half alone cost readability on *your* run? Whatever
the answer, what does this experiment tell you about exact metrics and optimisers?

In [ ]:
def fidelity_only(gold, pred) -> ev.ScoreReport:
    # TODO 1
    raise NotImplementedError

FIDELITY_BUDGET = 40
# TODO 2: tuned_fidelity, fidelity_gepa_cost
# TODO 3: c2
c2 = None

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=3):
    _p = lambda s: dspy.Prediction(sentence=s)
    _ex = ALL["giraffe-eats-only-leaves-de"]
    r = fidelity_only(_ex, _p("Jeder Giraffe frisst nur Blatt."))           # faithful, unreadable
    assert r.score == 1.0 and r.violated == []
    r = fidelity_only(_ex, _p("Giraffen fressen Blätter."))
    assert r.score == 0.0 and r.violated == ["template-for-only"]
    assert fidelity_only(_ex, _p("Jede Giraffe frisst mindestens ein Blatt.")).score == 0.0
    assert isinstance(c2, pd.DataFrame) and set(c2["program"]) == {"gepa", "gepa-fidelity-only"}
    assert {"fidelity", "full_score", "judge_acceptable", "eval_usd", "optimisation_usd"} <= set(c2.columns)
    assert c2[["fidelity", "full_score", "judge_acceptable"]].apply(lambda s: s.between(0, 1)).all().all()
    assert {"calls", "usd"} <= set(fidelity_gepa_cost) and FIDELITY_BUDGET <= 200

**Your answer:**

*(Write your answer here.)*

### C3. Is the difference bigger than the noise? (6 pts)

With caching **disabled** (`fresh = llm.dspy_lm(cache=False)`, `with dspy.context(lm=fresh)`)
run the baseline and the C1 tuned program on `test` three times each; store the mean
scores in `runs = {"baseline": [...], "gepa": [...]}`. Compute each program's mean and
sample standard deviation and set `verdict_c3 = "significant"` if
`|mean_gepa - mean_baseline| > 2 * max(sd_gepa, sd_baseline)`, else `"not significant"`.

In writing: what does this say about C1, and how does a task with an exact grader change
what "noise" means?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs (inside llm.meter(fresh) as c3_cost), then compute verdict_c3
verdict_c3 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=4):
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    _m = {k: st.mean(v) for k, v in runs.items()}
    _s = {k: st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["gepa"] - _m["baseline"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c3 == _expected
    assert {"calls", "usd"} <= set(c3_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part D — Knowing when to stop resampling

A verbaliser with a free exact check invites best-of-n: draw a sentence, check it, draw
again if it is not good enough. Each draw costs; the question is when to stop.
`A.RevisionMDP(qualities, probabilities, cost, max_attempts, recall)` is that decision:
the state is (attempts spent, quality in hand), the actions are `accept` and `retry`, a
new draft's quality is drawn from `probabilities`. With `recall=True` you keep the best
draft so far (a scorer ranks them); with `recall=False` only the latest.

### D1. Derive the stopping rule (8 pts)

1. Implement `expected_gain(q, qualities, probabilities)` = E[max(Q − q, 0)], the
   expected improvement of one more draw over a draft of quality `q` when you keep the
   better of the two.
2. Implement `reservation_policy(M)`: a policy table for every non-terminal state of `M`
   — `retry` at the start, `accept` when no attempts remain, and otherwise `retry` iff
   `expected_gain(q) > M.cost`. Derive on paper why this one-step rule is optimal for
   `recall=True` (hint: can the stopping condition, once met, ever stop holding?).
3. Implement `accept_threshold(M, policy) -> {attempts: lowest quality accepted}`.
   Using the ladder `D1_LADDER` below, build `d1` with columns `recall`, `attempts`,
   `threshold` from the value-iteration policies for `recall=True` and `recall=False`.

In writing: why is the threshold flat with recall and falling without it? Which of the
two describes (i) the DSPy verbaliser with the round trip as scorer, (ii) a keeper who
reviews one draft at a time?

In [ ]:
# The scorer's quality ladder: unparseable / wrong meaning / faithful with style issues / perfect.
D1_LADDER = dict(qualities=(0.0, 0.25, 0.75, 1.0), probabilities=(0.3, 0.2, 0.3, 0.2),
                 cost=0.02, max_attempts=6)
M = A.RevisionMDP(**D1_LADDER)
V, pi = mdp.value_iteration(M)
print(f"V*(start) = {V[M.initial_state()]:.4f}")
pd.DataFrame(M.thresholds(pi))

In [ ]:
def expected_gain(q: float, qualities, probabilities) -> float:
    # TODO 1
    raise NotImplementedError

def reservation_policy(M) -> dict:
    # TODO 2
    raise NotImplementedError

def accept_threshold(M, policy) -> dict:
    # TODO 3
    raise NotImplementedError

d1 = None        # TODO 3

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=5):
    assert abs(expected_gain(0.7, (0.4, 0.7, 1.0), (0.5, 0.3, 0.2)) - 0.06) < 1e-9
    assert expected_gain(1.0, (0.4, 0.7, 1.0), (0.5, 0.3, 0.2)) == 0.0
    for cost in np.linspace(0.0, 0.4, 17):
        for ladder in [((0.4, 0.7, 1.0), (0.5, 0.3, 0.2)), (D1_LADDER["qualities"], D1_LADDER["probabilities"])]:
            Mc = A.RevisionMDP(*ladder, cost=float(cost), max_attempts=5)
            _, pc = mdp.value_iteration(Mc)
            rule = reservation_policy(Mc)
            for s, action in pc.items():
                if 0 < s.attempts < Mc.max_attempts:
                    g = expected_gain(Mc.qualities[s.quality], *ladder)
                    if abs(g - cost) > 1e-6:
                        assert rule[s] == action, (cost, s, rule[s], action)
    assert isinstance(d1, pd.DataFrame) and {"recall", "attempts", "threshold"} <= set(d1.columns)
    for recall, grp in d1.sort_values("attempts").groupby("recall"):
        Mr = A.RevisionMDP(**D1_LADDER, recall=bool(recall))
        assert dict(zip(grp.attempts, grp.threshold)) == accept_threshold(Mr, mdp.value_iteration(Mr)[1])
        t = list(grp.threshold)
        assert all(a >= b for a, b in zip(t, t[1:])), "thresholds never rise as the budget runs out"
        assert t[-1] == min(D1_LADDER["qualities"]), "the last attempt accepts anything"
    _t = d1[d1.recall == True].sort_values("attempts").threshold.tolist()
    assert len(set(_t[:-1])) == 1, "with recall the threshold is flat until the cap"

**Your answer:**

*(Write your answer here.)*

### D2. Calibrate the stopping rule with live samples (8 pts)

The ladder in D1 was made up. Measure it.

1. With `fresh = llm.dspy_lm(cache=False)`, draw `K = 4` sentences per **test** item from
   `tuned` inside `llm.meter(fresh)` (cost in `d2_cost`) and score each with your scorer.
   Store `d2_samples` with columns `item`, `draw` (0..K−1), `score`.
2. `ladder` = the sorted distinct scores observed; `probs` = their relative frequencies.
3. `best_of_n` = list of length K: for n = 1..K, the mean over items of the best score
   among the first n draws.
4. Price a draw in quality units: `cost_per_draw = (d2_cost["usd"] / max(d2_cost["calls"], 1))
   / A.REVIEW_COST_USD`. Build `M_live = A.RevisionMDP(ladder, probs, cost_per_draw,
   max_attempts=K)`, solve it, and estimate `expected_draws` (mean number of draws of the
   optimal policy over 2 000 simulated episodes, `random.seed(0)`).

In writing: what does the live ladder look like, what does the rule recommend, and would
you trust `best_of_n` or the MDP more? What is the MDP assuming that your samples may
violate?

In [ ]:
K = 4
fresh = llm.dspy_lm(cache=False)
# TODO 1-4: d2_samples, d2_cost, ladder, probs, best_of_n, cost_per_draw, M_live, expected_draws

raise NotImplementedError

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=4):
    assert isinstance(d2_samples, pd.DataFrame) and len(d2_samples) == K * len(test)
    assert {"item", "draw", "score"} <= set(d2_samples.columns)
    assert set(d2_samples["item"]) == {ex.id for ex in test} and set(d2_samples["draw"]) == set(range(K))
    assert list(ladder) == sorted(set(d2_samples["score"]))
    assert abs(sum(probs) - 1) < 1e-9 and len(probs) == len(ladder)
    for q, p in zip(ladder, probs):
        assert abs(p - (d2_samples.score == q).mean()) < 1e-9
    assert len(best_of_n) == K and all(a <= b + 1e-12 for a, b in zip(best_of_n, best_of_n[1:]))
    assert abs(best_of_n[0] - d2_samples[d2_samples.draw == 0].score.mean()) < 1e-9
    assert {"calls", "usd"} <= set(d2_cost)
    assert abs(M_live.cost - (d2_cost["usd"] / max(d2_cost["calls"], 1)) / A.REVIEW_COST_USD) < 1e-12
    assert M_live.qualities == tuple(ladder) and M_live.max_attempts == K
    assert 1.0 <= expected_draws <= K

**Your answer:**

*(Write your answer here.)*

### D3. An agent that checks its own drafts (6 pts)

Give the job to an agent with the tools `A.build_toolset(ctx)` (`style_guide`,
`lookup_terms`, `check_round_trip`, `lint_sentence`).

1. Write `VERBALISER_PROMPT`: draft, **check the round trip and the lint**, redraft while
   either fails — at most the number of drafts your D2 rule recommends (say how you chose
   it) — and end with a JSON object `{"sentence": "...", "drafts": <int>}`.
2. Write `parse_sentence(text) -> str`: the `sentence` value of the last JSON object in the
   text that has one (code fences and surrounding prose allowed), else `""`.
3. For every **test** item, build a fresh `A.Ch9Context()`, tools and agent, run it, and
   collect `d3` with columns `id`, `language`, `sentence`, `score` (your scorer),
   `round_trip_checks` (number of `check_round_trip` calls), `tool_calls` and
   `usd_estimate` (`llm.chat_usage(run.messages)`).

In writing: compare the agent with the best-of-n program from D2 on score and cost per
sentence. Does its number of round-trip checks follow the stopping rule? What can the agent
check that the D2 loop cannot, and what can neither check?

In [ ]:
VERBALISER_PROMPT = "TODO"

def parse_sentence(text: str) -> str:
    # TODO
    raise NotImplementedError

d3 = None   # TODO: run the agent on every test item

In [ ]:
# Marking scheme for D3 -- do not edit.
with GRADER.check('D3', points=3):
    assert parse_sentence('{"sentence": "Kiara is a lion.", "drafts": 1}') == "Kiara is a lion."
    assert parse_sentence('Checked.\n```json\n{"sentence": "Jede Giraffe frisst nur Blätter.", "drafts": 2}\n```') \
        == "Jede Giraffe frisst nur Blätter."
    assert parse_sentence('{"note": 1} then {"sentence": "Elk insect is een dier.", "drafts": 1}') == "Elk insect is een dier."
    assert parse_sentence("Every keeper cares for at least one animal.") == ""
    assert parse_sentence("{}") == ""
    assert isinstance(d3, pd.DataFrame) and len(d3) == len(test)
    assert {"id", "language", "sentence", "score", "round_trip_checks", "tool_calls",
            "usd_estimate"} <= set(d3.columns)
    assert set(d3["id"]) == {ex.id for ex in test}
    assert d3["score"].between(0, 1).all() and (d3["round_trip_checks"] <= d3["tool_calls"]).all()
    for _, row in d3.iterrows():
        assert abs(row["score"] - verbalisation_scorer(ALL[row["id"]], dspy.Prediction(sentence=row["sentence"])).score) < 1e-9

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 Faithful is not the same as acceptable | 6 | 4 | 10 |
| A | A2 Audit the translation agency's delivery | 5 | 3 | 8 |
| B | B1 The staged scorer: exact half, style half | 8 | 4 | 12 |
| B | B2 The verbaliser as a DSPy program | 3 | 3 | 6 |
| B | B3 Baseline on the development split, with its cost | 2 | 4 | 6 |
| B | B4 Validate a readability judge against the keepers | 6 | 6 | 12 |
| C | C1 A budgeted GEPA run with a held-out report | 5 | 5 | 10 |
| C | C2 Optimise the exact half alone, and look at what you get | 3 | 5 | 8 |
| C | C3 Is the difference bigger than the noise? | 4 | 2 | 6 |
| D | D1 Derive the stopping rule | 5 | 3 | 8 |
| D | D2 Calibrate the stopping rule with live samples | 4 | 4 | 8 |
| D | D3 An agent that checks its own drafts | 3 | 3 | 6 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'B4', 'C1', 'C2', 'C3', 'D1', 'D2', 'D3']
_ = GRADER.summary(AUTO_GRADED)